In [2]:
from qdrant_client import QdrantClient
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings, Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter, SemanticSplitterNodeParser
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from pathlib import Path
import requests
import numpy as np
import torch
import gc
import re
from transformers import AutoTokenizer
import tqdm
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

/home/yuri/HSE/ВКР/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [145]:
COLLECT_NAME = "wb"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
CHUNK_SIZE = np.linspace(150,3000, 17, dtype=int).tolist() # в токенах
CHUNK_OVERLAP = 75
qwen_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)

In [146]:
len(CHUNK_SIZE)

17

In [147]:
# local_data 
local_data = Path.cwd() / "../data"

In [148]:
reader = SimpleDirectoryReader(input_dir=local_data, required_exts=[".pdf"])
docs = reader.load_data()
print(len(docs))
assert len(docs) == 86, "ОШИБКА, должно быть 86"

# full_text = "\n\n".join([d.text for d in docs])
# all_docs = Document(text=full_text)

86


In [149]:
# Сначала проставляем page_label ВСЕМ документам (до удаления!),
# чтобы номер страницы соответствовал реальному номеру в PDF
for i, doc in enumerate(docs):
    doc.metadata["page_label"] = i + 1

# Удаляем пустые, сохраняя правильные page_label
empty_indices = [i for i, doc in enumerate(docs) if len(doc.text) == 0]
print(f"Пустые страницы (реальные номера): {[docs[i].metadata['page_label'] for i in empty_indices]}")
for i in sorted(empty_indices, reverse=True):  # с конца, чтобы индексы не сдвигались
    docs.pop(i)

print(f"Документов после очистки: {len(docs)}")

Пустые страницы (реальные номера): [34, 86]
Документов после очистки: 84


In [150]:
len(docs)

84

In [151]:
# docs

In [152]:
def build_text_with_page_map(docs):
    """
    Объединяет документы в один текст и строит карту: 
    символьная позиция → номер страницы.
    
    Returns:
        full_text (str): объединённый текст всех документов
        page_ranges (list): [(start_char, end_char, page_label_str), ...]
    """
    full_text = ""
    page_ranges = []
    
    for doc in docs:
        start = len(full_text)
        full_text += doc.text + "\n\n"
        end = len(full_text)
        page_label = str(doc.metadata["page_label"])
        page_ranges.append((start, end, page_label))
    
    return full_text, page_ranges


# Использование:
full_text, page_ranges = build_text_with_page_map(docs)
all_docs = Document(text=full_text)

In [153]:
def get_pages_for_chunk(chunk_start, chunk_end, page_ranges):
    """
    По символьным позициям чанка определяет, к каким страницам он относится.
    Чанк «принадлежит» странице, если хотя бы часть его текста попала на эту страницу.
    """
    pages = []
    for pr_start, pr_end, page_label in page_ranges:
        if chunk_start < pr_end and chunk_end > pr_start:
            pages.append(page_label)
    return pages

In [154]:
def enrich_nodes_with_page_info(nodes, full_text, page_ranges):
    """
    Привязка страниц к чанкам через поиск позиции чанка в исходном тексте.
    Работает точно, без маркеров.
    """
    search_start = 0
    
    for node in nodes:
        idx = full_text.find(node.text, search_start)
        if idx == -1:
            # fallback: ищем с начала (может понадобиться из-за overlap)
            idx = full_text.find(node.text)
        
        if idx != -1:
            chunk_start = idx
            chunk_end = idx + len(node.text)
            node.metadata["pages_covered"] = get_pages_for_chunk(
                chunk_start, chunk_end, page_ranges
            )
            # Не сдвигаем на chunk_end, а на idx, из-за overlap
            search_start = idx
        else:
            # Крайне редкий случай, логируем для отладки
            print(f"⚠️ Чанк не найден в тексте: {node.text[:80]}...")
            node.metadata["pages_covered"] = []
    
    return nodes

In [155]:
def get_chunking(chunk_size, chunk_overlap, docs, full_text, page_ranges):
    splitter = TokenTextSplitter(chunk_size=chunk_size,
                                 chunk_overlap=chunk_overlap,
                                 tokenizer=qwen_tokenizer.encode)
    nodes = splitter.get_nodes_from_documents([docs])
    nodes = enrich_nodes_with_page_info(nodes, full_text, page_ranges)
    return nodes

In [ ]:
def create_collect(chunk_size, docs, full_text, page_ranges):
    client = QdrantClient(url="http://127.0.0.1:6333")

    col_name = f"{COLLECT_NAME}_token_spl_{chunk_size}_{CHUNK_OVERLAP}"

    if client.collection_exists(collection_name=col_name):
        client.delete_collection(collection_name=col_name)

    vector_store = QdrantVectorStore(collection_name=col_name,
                                     client=client)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    
    nodes = get_chunking(chunk_size, CHUNK_OVERLAP, docs, full_text, page_ranges)

    index = VectorStoreIndex(
        nodes=nodes,
        storage_context=storage_context,
        # show_progress=True
    )

    print(f"Коллекция {col_name} успешно создана и проиндексирована")

In [157]:
Settings.embed_model = OpenAILikeEmbedding(
    model_name="qwen3-embed",                    
    api_base="http://localhost:8081/v1",                      
)

try:
    req = requests.get(url="http://localhost:8081/health")
    if req.status_code == 200:
        print("✅ vLLM успешно подключён к LlamaIndex")
except Exception as e:
    print(f"Ошибка {e}")

for chunk_size in tqdm.tqdm(CHUNK_SIZE):
    create_collect(chunk_size, all_docs, full_text, page_ranges)

✅ vLLM успешно подключён к LlamaIndex


  6%|▌         | 1/17 [00:06<01:45,  6.62s/it]

Коллекция wb_token_spl_150_75 успешно создана и проиндексирована


 12%|█▏        | 2/17 [00:10<01:15,  5.03s/it]

Коллекция wb_token_spl_328_75 успешно создана и проиндексирована


...

Коллекция wb_token_spl_2643_75 успешно создана и проиндексирована


 94%|█████████▍| 16/17 [00:54<00:03,  3.10s/it]

Коллекция wb_token_spl_2821_75 успешно создана и проиндексирована


100%|██████████| 17/17 [00:57<00:00,  3.39s/it]

Коллекция wb_token_spl_3000_75 успешно создана и проиндексирована
